# Exhaustive LangGraph Agentic RAG


In [ ]:
# 1. Imports
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
import operator


In [ ]:
# 2. Define the State
# The state holds the memory of our entire graph execution.
class AgentState(TypedDict):
    question: str
    documents: list
    next_step: str


In [ ]:
# 3. Define the Nodes (Workers)
def route_question(state: AgentState):
    # Simulate an LLM deciding if the question is about company policy (RAG) or general facts (Web)
    if 'policy' in state['question'].lower():
        return {"next_step": "vector_search"}
    return {"next_step": "web_search"}

def vector_search(state: AgentState):
    # Simulate searching ChromaDB
    return {"documents": ["Company policy document"]}

def web_search(state: AgentState):
    # Simulate searching Google
    return {"documents": ["Google search result"]}


In [ ]:
# 4. Build the Graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("router", route_question)
workflow.add_node("vector_search", vector_search)
workflow.add_node("web_search", web_search)

# Add Conditional Edges
workflow.set_entry_point("router")
workflow.add_conditional_edges(
    "router",
    lambda x: x["next_step"],
    {
        "vector_search": "vector_search",
        "web_search": "web_search"
    }
)

workflow.add_edge("vector_search", END)
workflow.add_edge("web_search", END)

# 5. Compile and Run
app = workflow.compile()
# result = app.invoke({"question": "What is the remote work policy?"})
# print(result)
